# JJ resistance

Measures junction resistance with a ±13 mV I-V sweep, too small to change the junction. Needs `k2450.py` and `abaa.py` in this folder.

1. Run **Settings** and **Connect**. The output is then on at 0 V.
2. Land the needles, then run **Measure**.
3. For each junction: lift, re-land, measure again. The spread shows contact repeatability.
4. Run **Shut down** when you're done.

## Settings

In [4]:
from pathlib import Path
import pandas as pd

RESOURCE  = "TCPIP0::192.168.50.2::inst0::INSTR"
TERMINALS = "REAR"        # "FRON" for the front jacks
FOUR_WIRE = False
DATA_DIR  = Path(r"C:\Users\hatlab\Probing_Data\Israa")
SAMPLE    = "witness_batch_01"
ROOM_T_C  = None
R_SHORT   = 260.0         # Ohm: below this, flagged as shorted

## Connect

In [5]:
from K2450 import K2450
import ABAA as abaa

try:
    smu.close()
except Exception:
    pass

smu = K2450(RESOURCE, terminals=TERMINALS, four_wire=FOUR_WIRE)
smu.configure(ilim=1e-4)  # 100 uA: ample for a 13 mV readout, limits a short
smu.set_v(0.0)
smu.output(True)
abaa.setup(smu, sample=SAMPLE, data_dir=DATA_DIR, room_T_C=ROOM_T_C, r_short=R_SHORT)
print("Terminals:", smu.q(":ROUT:TERM?"), "  Output ON at 0 V")

KEITHLEY INSTRUMENTS,MODEL 2450,MYFP002618,1.7.16a
Terminals: REAR   Output ON at 0 V


## Measure
Change the name and rerun for each junction or re-land.

In [10]:
R = abaa.measure_R("JJ01")
if R < R_SHORT:
    print("SHORTED")

JJ01: R = 7940.62 Ohm  (fit residual 0.056 %)   n = 5, re-land spread 250.221 %


## Summary

In [11]:
summary = pd.DataFrame({jj: pd.Series(r) for jj, r in abaa.BASE.items()}).T
summary = summary.assign(
    mean=summary.mean(axis=1),
    spread_pct=(summary.max(axis=1) - summary.min(axis=1)) / summary.mean(axis=1) * 100,
)
summary.to_csv(DATA_DIR / f"{SAMPLE}_R_summary.csv")
summary

,0,1,2,3,4,mean,spread_pct
JJ01,9.470183,1.968768e+07,7945.969946,1.963701e+07,7940.621892,7.868118e+06,250.220838


## Shut down

In [12]:
smu.close()   # 0 V, output off